# Historical analysis notebook
Code-only copy. Outputs and original narrative cells were removed. See docs/reproduction.md for the revised analysis and execution order.


In [ ]:
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.base import clone
from sklearn.cross_decomposition import PLSRegression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.model_selection import GridSearchCV, KFold, LeaveOneOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
def locate_project_root() -> Path:
    start = Path.cwd().resolve()
    for root in [start, *start.parents]:
        if (root / 'data').exists() and (root / 'results').exists() and (root / 'images').exists():
            return root
    raise FileNotFoundError('Unable to locate project root')

PROJECT_ROOT = locate_project_root()
BASELINE_PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PAIRED_TARGET_PATH = PROJECT_ROOT / 'results' / 'q1_q2_vo2peak_abs_paired_table.csv'
RESULTS_DIR = PROJECT_ROOT / 'results'
IMAGES_DIR = PROJECT_ROOT / 'images'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMN = 'delta_vo2peak_abs'
TIME_WINDOW = (300.0, 1000.0)
WR_LEVELS = (30, 80)
MIN_SAMPLES_PER_LEVEL = 20

TIME_COLUMN_CANDIDATES = ('Time', 'time', 't')
WORKRATE_COLUMN_CANDIDATES = ('WorkRate', 'WR', 'work_rate', 'WR_cortex')
EXCLUDED_COLUMNS = {
    'Time', 'time', 't', 'Timecode',
    'WorkRate', 'workrate', 'WR', 'wr', 'work_rate', 'WR_cortex',
}

GAS_EXCHANGE_VENTILATION = [
    'VO2pulm', 'VO2_movMean', 'VCO2', 'FO2et', 'FCO2et', 'Vent', 'VT', 'BF',
]
HR_ONLY = ['HR']
HEMODYNAMICS = [
    'SV', 'CO', 'dPdt', 'SPTI', 'RPP', 'DPTI', 'DPTISPTI', 'LVET', 'Cwk',
    'Rp', 'TPR', 'fiSYS', 'fiMAP', 'fiDIA', 'sBP', 'mBP', 'dBP',
]
NIRS = ['absO2Hb_Rx1', 'absHHb_Rx1', 'TSI_Rx1_Tx1Tx2Tx3', 'TSI_FF_Rx1_Tx1Tx2Tx3']

ALL_SIGNALS = sorted({*HR_ONLY, *GAS_EXCHANGE_VENTILATION, *HEMODYNAMICS, *NIRS})
NO_GAS_EXCHANGE = sorted(set(ALL_SIGNALS) - set(GAS_EXCHANGE_VENTILATION))
HEMO_PLUS_NIRS = sorted(set(HEMODYNAMICS + NIRS))

SENSOR_GROUPS: Dict[str, Sequence[str]] = {
    'all_signals': ALL_SIGNALS,
    'no_gas_exchange': NO_GAS_EXCHANGE,
    'hemo_plus_nirs': HEMO_PLUS_NIRS,
    'nirs': NIRS,
}
GROUPS = ('all_signals', 'no_gas_exchange', 'hemo_plus_nirs', 'nirs')
MODELS = ('Ridge', 'ElasticNet', 'PLS')
GROUP_DISPLAY = {
    'all_signals': 'All signals',
    'no_gas_exchange': 'No gas exchange',
    'hemo_plus_nirs': 'Hemodynamics + NIRS',
    'nirs': 'NIRS only',
}

PROJECT_ROOT

In [ ]:
def safe_name(value: str) -> str:
    return value.replace(' ', '_').replace('/', '_')

def detect_column(columns: Iterable[str], candidates: Sequence[str]) -> str:
    lower_map = {col.lower(): col for col in columns}
    for candidate in candidates:
        key = candidate.lower()
        if key in lower_map:
            return lower_map[key]
    raise KeyError(f'Could not find any candidate columns in {candidates}')

def load_baseline_subject(subject_id: str) -> pd.DataFrame:
    path = BASELINE_PROCESSED_DIR / f'{subject_id}_data.csv'
    if not path.exists():
        raise FileNotFoundError(f'Missing baseline file for {subject_id}: {path}')
    return pd.read_csv(path)

def slice_prbs_window(df: pd.DataFrame, time_col: str, window: Tuple[float, float]) -> pd.DataFrame:
    start, stop = window
    out = df[(df[time_col] >= start) & (df[time_col] <= stop)].copy()
    if out.empty:
        raise ValueError(f'No rows in PRBS window {window}')
    return out

def select_numeric_signals(df: pd.DataFrame, candidates: Sequence[str]) -> List[str]:
    numeric_cols = set(df.select_dtypes(include=[np.number]).columns)
    return [col for col in candidates if col in numeric_cols]

def extract_wr_conditioned_features(
    df: pd.DataFrame,
    signal_columns: Sequence[str],
    wr_column: str,
    min_samples_per_level: int,
    wr_levels: Tuple[int, int],
) -> Dict[str, float]:
    feats: Dict[str, float] = {}
    lower_wr, upper_wr = sorted(wr_levels)

    for signal in signal_columns:
        mean_lower = np.nan
        mean_upper = np.nan
        for wr in wr_levels:
            wr_values = df.loc[df[wr_column] == wr, signal].dropna()
            mean_name = f'{signal}_mean_wr{wr}'
            std_name = f'{signal}_std_wr{wr}'
            feats[mean_name] = np.nan
            feats[std_name] = np.nan
            if wr_values.shape[0] >= min_samples_per_level:
                feats[mean_name] = float(wr_values.mean())
                feats[std_name] = float(wr_values.std(ddof=1))
                if wr == lower_wr:
                    mean_lower = feats[mean_name]
                else:
                    mean_upper = feats[mean_name]

        delta_name = f'{signal}_mean_delta_wr{upper_wr}_minus_wr{lower_wr}'
        feats[delta_name] = (
            mean_upper - mean_lower
            if not np.isnan(mean_lower) and not np.isnan(mean_upper)
            else np.nan
        )

    return feats

def build_baseline_feature_table(
    subject_ids: Sequence[str],
    group_columns: Sequence[str],
) -> pd.DataFrame:
    rows: List[Dict[str, float]] = []
    for sid in subject_ids:
        data_df = load_baseline_subject(sid)
        time_col = detect_column(data_df.columns, TIME_COLUMN_CANDIDATES)
        wr_col = detect_column(data_df.columns, WORKRATE_COLUMN_CANDIDATES)
        prbs_df = slice_prbs_window(data_df, time_col, TIME_WINDOW)

        if group_columns == ALL_SIGNALS:
            candidates = [col for col in prbs_df.columns if col not in EXCLUDED_COLUMNS]
            signal_columns = select_numeric_signals(prbs_df, candidates)
        else:
            signal_columns = select_numeric_signals(prbs_df, group_columns)

        feats = {'subject_id': sid}
        if signal_columns:
            feats.update(
                extract_wr_conditioned_features(
                    prbs_df,
                    signal_columns,
                    wr_column=wr_col,
                    min_samples_per_level=MIN_SAMPLES_PER_LEVEL,
                    wr_levels=WR_LEVELS,
                )
            )
        rows.append(feats)

    out = pd.DataFrame(rows).set_index('subject_id').sort_index()
    return out

def make_model_search(model_name: str, n_train: int, n_features: int) -> GridSearchCV:
    if model_name == 'Ridge':
        estimator = Ridge(random_state=42)
        param_grid = {'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0, 300.0]}
    elif model_name == 'ElasticNet':
        estimator = ElasticNet(random_state=42, max_iter=10000)
        param_grid = {
            'model__alpha': [0.001, 0.01, 0.1, 1.0, 10.0],
            'model__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9],
        }
    elif model_name == 'PLS':
        max_comp = int(min(8, n_train - 1, n_features))
        if max_comp < 1:
            raise ValueError('Invalid n_components for PLS in current fold')
        estimator = PLSRegression()
        param_grid = {'model__n_components': list(range(1, max_comp + 1))}
    else:
        raise ValueError(f'Unknown model: {model_name}')

    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', estimator),
    ])

    n_inner = max(2, min(5, n_train))
    inner_cv = KFold(n_splits=n_inner, shuffle=True, random_state=42)

    return GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring='neg_mean_absolute_error',
        cv=inner_cv,
        n_jobs=-1,
        refit=True,
    )

def summarize_delta_metrics(actual: np.ndarray, pred: np.ndarray) -> Dict[str, float]:
    residual = pred - actual
    mae = float(np.mean(np.abs(residual)))
    rmse = float(np.sqrt(np.mean(residual ** 2)))
    ss_res = float(np.sum((actual - pred) ** 2))
    ss_tot = float(np.sum((actual - np.mean(actual)) ** 2))
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else np.nan

    if len(actual) >= 2 and np.std(actual, ddof=1) > 0 and np.std(pred, ddof=1) > 0:
        pearson_r, pearson_p = stats.pearsonr(pred, actual)
    else:
        pearson_r, pearson_p = np.nan, np.nan

    if len(actual) >= 2:
        spearman_rho, spearman_p = stats.spearmanr(pred, actual)
    else:
        spearman_rho, spearman_p = np.nan, np.nan

    if len(actual) >= 2 and np.std(pred, ddof=1) > 0:
        calib = stats.linregress(pred, actual)
        cal_slope, cal_intercept = float(calib.slope), float(calib.intercept)
    else:
        cal_slope, cal_intercept = np.nan, np.nan

    if len(actual) >= 2 and not np.allclose(residual, 0.0):
        t_stat, t_p = stats.ttest_rel(pred, actual)
        try:
            w_stat, w_p = stats.wilcoxon(pred, actual, zero_method='wilcox', alternative='two-sided')
        except ValueError:
            w_stat, w_p = np.nan, np.nan
        sd_res = float(np.std(residual, ddof=1))
        dz = float(np.mean(residual) / sd_res) if sd_res > 0 else np.nan
    else:
        t_stat, t_p, w_stat, w_p, dz = np.nan, np.nan, np.nan, np.nan, np.nan

    sign_agreement = float(100.0 * np.mean(np.sign(actual) == np.sign(pred)))

    return {
        'MAE_delta': mae,
        'RMSE_delta': rmse,
        'R2_delta': r2,
        'MeanResidual_delta': float(np.mean(residual)),
        'StdResidual_delta': float(np.std(residual, ddof=1)) if len(actual) > 1 else np.nan,
        'Pearson_r': float(pearson_r) if not np.isnan(pearson_r) else np.nan,
        'Pearson_p': float(pearson_p) if not np.isnan(pearson_p) else np.nan,
        'Spearman_rho': float(spearman_rho) if not np.isnan(spearman_rho) else np.nan,
        'Spearman_p': float(spearman_p) if not np.isnan(spearman_p) else np.nan,
        'Calibration_slope': cal_slope,
        'Calibration_intercept': cal_intercept,
        'PairedT_stat': float(t_stat) if not np.isnan(t_stat) else np.nan,
        'PairedT_p': float(t_p) if not np.isnan(t_p) else np.nan,
        'Wilcoxon_stat': float(w_stat) if not np.isnan(w_stat) else np.nan,
        'Wilcoxon_p': float(w_p) if not np.isnan(w_p) else np.nan,
        'Cohens_dz': dz,
        'SignAgreementPct': sign_agreement,
    }

In [ ]:
paired_df = pd.read_csv(PAIRED_TARGET_PATH)
if TARGET_COLUMN not in paired_df.columns:
    raise KeyError(f'Missing target column {TARGET_COLUMN} in {PAIRED_TARGET_PATH.name}')

paired_df = paired_df[['subject_id', 'VO2peak_abs_pre', 'VO2peak_abs_post', TARGET_COLUMN]].copy()
paired_df['subject_id'] = paired_df['subject_id'].astype(str)
paired_df = paired_df.set_index('subject_id').sort_index()

available_subjects = []
for sid in paired_df.index:
    if (BASELINE_PROCESSED_DIR / f'{sid}_data.csv').exists():
        available_subjects.append(sid)

if not available_subjects:
    raise ValueError('No paired subjects with baseline processed files found.')

paired_df = paired_df.loc[available_subjects].copy()
subject_ids = paired_df.index.tolist()
y_delta = paired_df[TARGET_COLUMN].copy()

summary_rows: List[Dict[str, float]] = []
subject_rows: List[Dict[str, float]] = []

for group in GROUPS:
    X = build_baseline_feature_table(subject_ids, SENSOR_GROUPS[group]).reindex(subject_ids)

    for model_name in MODELS:
        loo = LeaveOneOut()
        pred_records: List[Dict[str, float]] = []
        param_records: List[Dict[str, str]] = []

        for fold_idx, (train_idx, test_idx) in enumerate(loo.split(X), start=1):
            sid_test = subject_ids[test_idx[0]]
            X_train = X.iloc[train_idx]
            y_train = y_delta.iloc[train_idx]
            X_test = X.iloc[[test_idx[0]]]
            y_test = float(y_delta.iloc[test_idx[0]])

            search = make_model_search(
                model_name=model_name,
                n_train=len(train_idx),
                n_features=X_train.shape[1],
            )
            search.fit(X_train, y_train)
            y_pred = float(search.best_estimator_.predict(X_test)[0])

            pred_records.append({
                'subject_id': sid_test,
                'actual_delta': y_test,
                'predicted_delta': y_pred,
                'residual_delta': y_pred - y_test,
                'abs_error_delta': abs(y_pred - y_test),
            })

            param_records.append({
                'subject_id': sid_test,
                'outer_fold': fold_idx,
                'best_params': str(search.best_params_),
                'best_score_neg_mae': float(search.best_score_),
            })

        pred_df = pd.DataFrame(pred_records).set_index('subject_id').sort_index()
        params_df = pd.DataFrame(param_records).sort_values('outer_fold').reset_index(drop=True)

        token = f"e4_baseline_to_delta_{safe_name(group)}_{safe_name(model_name)}"
        pred_path = RESULTS_DIR / f'{token}_predictions.csv'
        params_path = RESULTS_DIR / f'{token}_selected_params.csv'
        pred_df.to_csv(pred_path)
        params_df.to_csv(params_path, index=False)

        actual = pred_df['actual_delta'].to_numpy(dtype=float)
        pred = pred_df['predicted_delta'].to_numpy(dtype=float)
        metrics = summarize_delta_metrics(actual, pred)

        summary_rows.append({
            'Group': group,
            'GroupLabel': GROUP_DISPLAY[group],
            'Model': model_name,
            'NSubjects': int(len(pred_df)),
            'NFeatures': int(X.shape[1]),
            **metrics,
        })

        for sid, row in pred_df.iterrows():
            subject_rows.append({
                'subject_id': sid,
                'Group': group,
                'GroupLabel': GROUP_DISPLAY[group],
                'Model': model_name,
                'actual_delta': float(row['actual_delta']),
                'predicted_delta': float(row['predicted_delta']),
                'residual_delta': float(row['residual_delta']),
                'abs_error_delta': float(row['abs_error_delta']),
            })

e4_summary_df = pd.DataFrame(summary_rows)
e4_subject_df = pd.DataFrame(subject_rows)

if e4_summary_df.empty:
    raise ValueError('No E4 summaries were produced.')

group_order = {name: idx for idx, name in enumerate(GROUPS)}
model_order = {name: idx for idx, name in enumerate(MODELS)}
e4_summary_df['GroupOrder'] = e4_summary_df['Group'].map(group_order).fillna(999)
e4_summary_df['ModelOrder'] = e4_summary_df['Model'].map(model_order).fillna(999)
e4_summary_df = e4_summary_df.sort_values(['GroupOrder', 'ModelOrder']).reset_index(drop=True)

summary_path = RESULTS_DIR / 'e4_baseline_to_delta_summary.csv'
subject_path = RESULTS_DIR / 'e4_baseline_to_delta_subject_level.csv'
e4_summary_df.to_csv(summary_path, index=False)
e4_subject_df.to_csv(subject_path, index=False)

display_cols = [
    'GroupLabel', 'Model', 'NSubjects', 'NFeatures',
    'MAE_delta', 'RMSE_delta', 'R2_delta',
    'Pearson_r', 'Pearson_p', 'SignAgreementPct',
    'PairedT_p', 'Wilcoxon_p',
]
display(e4_summary_df[display_cols])
print('Saved:', summary_path)
print('Saved:', subject_path)

In [ ]:
table_full = e4_summary_df[[
    'GroupLabel', 'Model', 'NSubjects', 'NFeatures',
    'MAE_delta', 'RMSE_delta', 'R2_delta',
    'MeanResidual_delta', 'StdResidual_delta',
    'Pearson_r', 'Pearson_p', 'Spearman_rho', 'Spearman_p',
    'Calibration_slope', 'Calibration_intercept',
    'SignAgreementPct', 'PairedT_p', 'Wilcoxon_p', 'Cohens_dz',
]].rename(columns={
    'GroupLabel': 'Group',
    'NSubjects': 'N',
    'MeanResidual_delta': 'Delta bias (pred-obs)',
    'StdResidual_delta': 'Delta resid SD',
    'SignAgreementPct': 'Sign agreement (%)',
    'PairedT_p': 'Paired t p',
    'Wilcoxon_p': 'Wilcoxon p',
})

table_full_csv = RESULTS_DIR / 'e4_results_table_full.csv'
table_full_tex = RESULTS_DIR / 'e4_results_table_full.tex'
table_full.to_csv(table_full_csv, index=False)

table_tex_df = table_full.copy()
num_cols = [
    'MAE_delta', 'RMSE_delta', 'R2_delta',
    'Delta bias (pred-obs)', 'Delta resid SD',
    'Pearson_r', 'Spearman_rho',
    'Calibration_slope', 'Calibration_intercept',
    'Sign agreement (%)', 'Cohens_dz',
]
for col in num_cols:
    table_tex_df[col] = table_tex_df[col].map(lambda x: round(float(x), 3) if pd.notna(x) else np.nan)
for col in ['Pearson_p', 'Spearman_p', 'Paired t p', 'Wilcoxon p']:
    table_tex_df[col] = table_full[col].map(
        lambda x: '<0.001' if pd.notna(x) and float(x) < 0.001 else (f'{float(x):.3f}' if pd.notna(x) else 'NA')
    )

table_tex_df.to_latex(
    table_full_tex,
    index=False,
    escape=False,
    caption='Experiment 4: baseline-to-delta modeling performance across all feature-group/model combinations.',
    label='tab:e4_full',
)

best_by_group = e4_summary_df.sort_values('MAE_delta').groupby('Group', as_index=False).first()
best_table = best_by_group[[
    'GroupLabel', 'Model', 'NSubjects',
    'MAE_delta', 'RMSE_delta', 'R2_delta',
    'Pearson_r', 'Pearson_p',
    'SignAgreementPct', 'PairedT_p', 'Wilcoxon_p',
]].rename(columns={
    'GroupLabel': 'Group',
    'NSubjects': 'N',
    'SignAgreementPct': 'Sign agreement (%)',
    'PairedT_p': 'Paired t p',
    'Wilcoxon_p': 'Wilcoxon p',
})

best_csv = RESULTS_DIR / 'e4_results_table_best_by_group.csv'
best_tex = RESULTS_DIR / 'e4_results_table_best_by_group.tex'
best_table.to_csv(best_csv, index=False)

best_tex_df = best_table.copy()
for col in ['MAE_delta', 'RMSE_delta', 'R2_delta', 'Pearson_r', 'Sign agreement (%)']:
    best_tex_df[col] = best_tex_df[col].map(lambda x: round(float(x), 3) if pd.notna(x) else np.nan)
for col in ['Pearson_p', 'Paired t p', 'Wilcoxon p']:
    best_tex_df[col] = best_table[col].map(
        lambda x: '<0.001' if pd.notna(x) and float(x) < 0.001 else (f'{float(x):.3f}' if pd.notna(x) else 'NA')
    )

best_tex_df.to_latex(
    best_tex,
    index=False,
    escape=False,
    caption='Experiment 4: best model within each feature group for baseline-to-delta prediction.',
    label='tab:e4_best_group',
)

display(table_tex_df)
print('Saved:', table_full_csv)
print('Saved:', table_full_tex)
print('Saved:', best_csv)
print('Saved:', best_tex)

In [ ]:
# Figure 1: Delta MAE heatmap
delta_mae_matrix = e4_summary_df.pivot(index='Group', columns='Model', values='MAE_delta').reindex(GROUPS)[list(MODELS)]
delta_mae_vals = delta_mae_matrix.to_numpy(dtype=float)

fig, ax = plt.subplots(figsize=(7.2, 3.4))
im = ax.imshow(delta_mae_vals, cmap='viridis_r', aspect='auto')
ax.set_xticks(np.arange(len(MODELS)), labels=MODELS)
ax.set_yticks(np.arange(len(GROUPS)), labels=[GROUP_DISPLAY[g] for g in GROUPS])
threshold = np.nanmedian(delta_mae_vals)
for i in range(delta_mae_vals.shape[0]):
    for j in range(delta_mae_vals.shape[1]):
        value = delta_mae_vals[i, j]
        color = 'white' if value > threshold else 'black'
        ax.text(j, i, f'{value:.3f}', ha='center', va='center', color=color, fontsize=9)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Delta MAE [L/min]')
ax.set_title('E4 baseline-to-delta performance heatmap')
ax.set_xlabel('Model')
ax.set_ylabel('Feature group')
fig.tight_layout()
fig1_path = IMAGES_DIR / 'e4_delta_mae_heatmap.png'
fig.savefig(fig1_path, dpi=300, bbox_inches='tight')
plt.close(fig)

# Best combo
best_row = e4_summary_df.sort_values('MAE_delta').iloc[0]
best_group = best_row['Group']
best_model = best_row['Model']
best_subject = e4_subject_df[
    (e4_subject_df['Group'] == best_group) & (e4_subject_df['Model'] == best_model)
].copy()

# Figure 2: Predicted vs observed delta (best combo)
x = best_subject['actual_delta'].to_numpy(dtype=float)
y = best_subject['predicted_delta'].to_numpy(dtype=float)
lims = [float(min(x.min(), y.min())), float(max(x.max(), y.max()))]

fig, ax = plt.subplots(figsize=(5.2, 5.0))
ax.scatter(x, y, color='navy', alpha=0.85)
ax.plot(lims, lims, '--', color='gray', linewidth=1, label='Ideal 1:1')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel('Observed delta VO2peak [L/min]')
ax.set_ylabel('Predicted delta VO2peak [L/min]')
ax.set_title(f"E4 best: {GROUP_DISPLAY[best_group]} + {best_model}")
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend()
fig.tight_layout()
fig2_path = IMAGES_DIR / 'e4_actual_vs_pred_delta_best.png'
fig.savefig(fig2_path, dpi=300, bbox_inches='tight')
plt.close(fig)

# Figure 3: Bland-Altman (best combo)
mean_vals = 0.5 * (x + y)
diff_vals = y - x
mean_diff = float(np.mean(diff_vals))
sd_diff = float(np.std(diff_vals, ddof=1)) if len(diff_vals) > 1 else np.nan
loa = 1.96 * sd_diff if pd.notna(sd_diff) else np.nan

fig, ax = plt.subplots(figsize=(6.0, 4.0))
ax.scatter(mean_vals, diff_vals, color='darkorange', alpha=0.85)
ax.axhline(mean_diff, color='black', linestyle='--', linewidth=1, label='Mean bias')
if pd.notna(loa):
    ax.axhline(mean_diff + loa, color='red', linestyle='--', linewidth=0.9, label='Upper LOA')
    ax.axhline(mean_diff - loa, color='red', linestyle='--', linewidth=0.9, label='Lower LOA')
ax.set_xlabel('Mean of observed/predicted delta [L/min]')
ax.set_ylabel('Delta error (pred - obs) [L/min]')
ax.set_title(f"E4 Bland-Altman: {GROUP_DISPLAY[best_group]} + {best_model}")
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(loc='upper right')
fig.tight_layout()
fig3_path = IMAGES_DIR / 'e4_delta_bland_altman_best.png'
fig.savefig(fig3_path, dpi=300, bbox_inches='tight')
plt.close(fig)

print('Best combo by delta MAE:', GROUP_DISPLAY[best_group], '+', best_model)
print('Saved:', fig1_path)
print('Saved:', fig2_path)
print('Saved:', fig3_path)

In [ ]:
def fmt_p(p: float) -> str:
    if pd.isna(p):
        return 'NA'
    return '<0.001' if float(p) < 0.001 else f'{float(p):.3f}'

e4_sorted = e4_summary_df.sort_values('MAE_delta').reset_index(drop=True)
best_cfg = e4_sorted.iloc[0]
worst_cfg = e4_sorted.iloc[-1]
mean_sign_agreement = float(e4_summary_df['SignAgreementPct'].mean())
n_signif_t = int((e4_summary_df['PairedT_p'] < 0.05).sum())
n_signif_w = int((e4_summary_df['Wilcoxon_p'] < 0.05).sum())

paragraph_1 = (
    f"Experiment 4 directly modeled training-induced VO2peak change from baseline PRBS-derived "
    f"features using LOOCV in the paired follow-up cohort ($n={int(best_cfg['NSubjects'])}$). "
    f"Across 12 feature-group/model combinations, delta-prediction MAE ranged from "
    f"{best_cfg['MAE_delta']:.3f} to {worst_cfg['MAE_delta']:.3f}~L$\\cdot$min$^{{-1}}$ "
    f"(Table~\\ref{{tab:e4_full}})."
 )

paragraph_2 = (
    f"The best configuration was {best_cfg['GroupLabel']} + {best_cfg['Model']} "
    f"(MAE$_\\Delta$={best_cfg['MAE_delta']:.3f}, RMSE$_\\Delta$={best_cfg['RMSE_delta']:.3f}, "
    f"$R^2_\\Delta$={best_cfg['R2_delta']:.3f}, Pearson $r$={best_cfg['Pearson_r']:.3f}, "
    f"sign agreement={best_cfg['SignAgreementPct']:.1f}\\%). "
    f"The worst configuration was {worst_cfg['GroupLabel']} + {worst_cfg['Model']} "
    f"(MAE$_\\Delta$={worst_cfg['MAE_delta']:.3f})."
 )

paragraph_3 = (
    f"Across configurations, mean sign agreement was {mean_sign_agreement:.1f}\\%, with "
    f"{n_signif_t}/12 paired t-tests and {n_signif_w}/12 Wilcoxon tests significant at $p<0.05$. "
    f"This suggests variable but measurable baseline-to-delta predictive utility depending on "
    f"signal subset and model class."
 )

paragraph_4 = (
    "Figure~\\ref{fig:e4_delta_mae_heatmap} shows the delta-MAE landscape across configurations. "
    "Figure~\\ref{fig:e4_scatter_best} and Figure~\\ref{fig:e4_ba_best} provide agreement and "
    "Bland--Altman views for the best configuration. Table~\\ref{tab:e4_best_group} "
    "summarizes the top model in each feature group."
 )

section_text = "\n\n".join([
    r"\subsection{E4: Baseline-to-delta modeling}",
    paragraph_1,
    paragraph_2,
    paragraph_3,
    paragraph_4,
]) + "\n"

figures_text = """\\begin{figure}[t]
\\centering
\\includegraphics[width=0.95\\linewidth]{images/e4_delta_mae_heatmap.png}
\\caption{Experiment 4 delta-prediction MAE heatmap across feature groups and models (lower is better).}
\\label{fig:e4_delta_mae_heatmap}
\\end{figure}

\\begin{figure}[t]
\\centering
\\includegraphics[width=0.95\\linewidth]{images/e4_actual_vs_pred_delta_best.png}
\\caption{Predicted versus observed VO2peak change for the best E4 configuration.}
\\label{fig:e4_scatter_best}
\\end{figure}

\\begin{figure}[t]
\\centering
\\includegraphics[width=0.95\\linewidth]{images/e4_delta_bland_altman_best.png}
\\caption{Bland--Altman analysis for VO2peak change in the best E4 configuration.}
\\label{fig:e4_ba_best}
\\end{figure}
"""

bundle_text = (
    section_text
    + "\n% Auto-generated tables\n"
    + "\\input{results/e4_results_table_full.tex}\n"
    + "\\input{results/e4_results_table_best_by_group.tex}\n\n"
    + figures_text
 )

section_path = RESULTS_DIR / 'e4_results_section.tex'
bundle_path = RESULTS_DIR / 'e4_results_with_tables_and_figures.tex'
section_path.write_text(section_text, encoding='utf-8')
bundle_path.write_text(bundle_text, encoding='utf-8')

print(section_text)
print('Saved:', section_path)
print('Saved:', bundle_path)